In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

In [2]:
panel_final = pd.read_csv("data/df_model_panel.csv")

In [3]:
id_cols = ["country", "country_code", "year"]

features = [
    c for c in panel_final.columns
    if c not in id_cols
]

In [4]:
scaler = StandardScaler()

X = scaler.fit_transform(
    panel_final[features]
)

In [5]:
pca = PCA(n_components=7)

X_pca = pca.fit_transform(X)


In [6]:
explained = pd.DataFrame({
    "PC": np.arange(1, pca.n_components_ + 1),
    "Explained_Variance": pca.explained_variance_ratio_,
    "Cumulative": pca.explained_variance_ratio_.cumsum()
})


print(explained)

   PC  Explained_Variance  Cumulative
0   1            0.221047    0.221047
1   2            0.158038    0.379086
2   3            0.096485    0.475570
3   4            0.079053    0.554623
4   5            0.072497    0.627119
5   6            0.066944    0.694063
6   7            0.053978    0.748042


In [7]:
panel_final["fiscal_risk_index"] = X_pca[:, 0]

In [8]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=features,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)

print(loadings)

                                      PC1       PC2       PC3       PC4  \
gdp_per_capita                   0.362960  0.209088  0.218937  0.055278   
real_gdp_growth                 -0.121507  0.278128 -0.001629 -0.137697   
inflation_rate                  -0.115457 -0.214067  0.593230  0.037800   
compulsory_education_years       0.199745 -0.051568 -0.023545  0.602713   
population_65_plus_pct           0.449711 -0.033613  0.039528  0.005286   
unemployment_rate                0.030508 -0.264198 -0.186255 -0.388347   
government_expenditure_pct_gdp   0.422177 -0.221862 -0.132082 -0.111736   
government_revenue_pct_gdp       0.434474 -0.025746 -0.033495 -0.212115   
gdp_deflator_inflation          -0.103333 -0.176295  0.602760 -0.038984   
age_dependency_ratio            -0.297289 -0.158178 -0.000524 -0.303899   
gross_capital_formation_pct_gdp -0.125216  0.334086 -0.177638 -0.157138   
total_reserves_months_imports   -0.104317  0.087416 -0.101137  0.479745   
savings_pct_gdp          

In [9]:
weights = (
    pca.explained_variance_ratio_
    / pca.explained_variance_ratio_.sum()
)

panel_final["fiscal_risk_index"] = (
    X_pca * weights
).sum(axis=1)

In [10]:
scaler = MinMaxScaler(feature_range=(0, 100))

panel_final["fiscal_risk_index_100"] = (
    scaler.fit_transform(
        panel_final[["fiscal_risk_index"]]
    )
)

In [11]:
panel_final.to_csv(
    "data/fiscal_risk_index.csv",
    index=False
)